# Introduction

Exploratory Data Analysis (EDA) characterises the data before modelling: its quality, the shape of each variable, which variables separate at-risk from not-at-risk students, how variables relate to one another, and — because this is a *time-aware* problem — when the predictive signal emerges. These observations directly inform the three research questions: the earliest reliable checkpoint and best algorithm (**RQ1**), the features whose importance SHAP/LIME should later explain stably (**RQ2**), and the class balance that motivates the imbalance study (**RQ3**).

# Data and statistical method

The analysis uses the master table (t = 100%) for univariate, bivariate and correlation analysis, and the six checkpoint datasets for the time-aware analysis. To move beyond visual impression we apply: the **Mann-Whitney U** test (non-parametric, appropriate for the right-skewed features) with **Benjamini-Hochberg** false-discovery-rate correction and **Cohen's d** effect size for numeric-vs-target comparisons; the **chi-square** test of independence with **Cramer's V** effect size for categorical-vs-target association; and **Pearson/Spearman** correlation for multivariate structure. All charts follow the team chart standard (`docs/standards`).

In [ ]:
import sys, json
from pathlib import Path
import pandas as pd
from IPython.display import Image, display
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
TABLES = ROOT / 'reports' / 'tables'
FIGS = ROOT / 'reports' / 'figures'
from src.eda import eda
master = eda.load_master()
checkpoints = eda.load_checkpoints()
print('master:', master.shape, '| checkpoints stacked:', None if checkpoints is None else checkpoints.shape)

# Data quality

Only three columns contain missing values: `date_unregistration` (structurally absent for students who never withdraw — not a feature), `imd_band` (1,111; filled `Unknown`), and `date_registration` (45; train-median imputed). No feature column is materially incomplete.

In [ ]:
print(json.dumps(eda.data_quality(master), indent=2))
display(Image(filename=str(FIGS / 'quality_missingness.png')))

# Univariate analysis

Engagement (clickstream) features are strongly right-skewed and heavy-tailed (`clicks_resource` skew ≈ 35, `max_clicks_single_day` ≈ 11), which justifies the `log1p` transform applied during cleaning. Categorical features are well populated across their levels.

In [ ]:
# Table — numeric descriptive statistics (with skew & kurtosis)
print(json.dumps(eda.univariate(master), indent=2, ensure_ascii=False))
display(pd.read_csv(TABLES / 'univariate_numeric.csv', index_col=0))

In [ ]:
display(Image(filename=str(FIGS / 'univariate_hist_kde.png')))

In [ ]:
display(Image(filename=str(FIGS / 'univariate_boxplots.png')))

In [ ]:
display(Image(filename=str(FIGS / 'univariate_categorical_freq.png')))

# Target distribution and class imbalance (STT 27)

The observed at-risk rate is **52.8%** (imbalance ratio 1.12): a *slight majority*, not the 68/32 shown illustratively on the course slides. Imbalance is therefore mild — which we report honestly and which frames **RQ3** (resampling may yield only modest gains). PR-AUC and recall on the at-risk class remain the headline metrics because a missed at-risk student is the costly error.

In [ ]:
print(json.dumps(eda.target_distribution(master), indent=2))
display(Image(filename=str(FIGS / 'target_distribution.png')))

# Bivariate analysis — numeric features vs the target (STT 36)

For each numeric feature we test whether its distribution differs by class (Mann-Whitney U, BH-corrected) and quantify the gap (Cohen's d). **All 19 features are significant** (q < 0.05) — expected at n ≈ 32k — so effect size, not the p-value, is the discriminator. The strongest are behavioural/performance: `days_since_last_activity` (d = 2.55), `n_assessments_submitted` (2.05) and `weighted_score_to_date` (1.96). Demographic features are weakest (`studied_credits` 0.28, `num_of_prev_attempts` 0.21), empirically reproducing the prior-work finding that behaviour outweighs demographics.

In [ ]:
# Table — effect size + Mann-Whitney U (BH-corrected) per numeric feature
print(json.dumps(eda.numeric_vs_target(master), indent=2))
display(pd.read_csv(TABLES / 'bivariate_numeric_tests.csv'))

In [ ]:
display(Image(filename=str(FIGS / 'bivariate_effect_sizes.png')))

In [ ]:
display(Image(filename=str(FIGS / 'bivariate_top_boxplots.png')))

# Bivariate analysis — categorical features vs the target

Chi-square tests are significant but the **Cramer's V** effect sizes are small: `highest_education` (0.15) and `imd_band` (0.15) are the strongest demographic associations, while `gender` (0.02) is negligible. This confirms demographics carry limited standalone signal and are best retained for fairness analysis rather than predictive reliance.

In [ ]:
print(json.dumps(eda.categorical_vs_target(master), indent=2))
display(pd.read_csv(TABLES / 'bivariate_categorical_tests.csv'))
display(Image(filename=str(FIGS / 'bivariate_categorical_rate.png')))

# Multivariate analysis — correlation, multicollinearity, leakage (STT 37)

Pearson and Spearman matrices agree on the structure. Two pairs are multicollinear (|r| ≥ 0.8): `n_days_active`–`total_clicks` (0.84) and `days_since_last_activity`–`n_assessments_submitted` (−0.83); this is relevant to explanation stability (**RQ2**), since SHAP/LIME may distribute importance among correlated features. Critically, **no feature correlates ≥ 0.95 with the target**, so no leakage feature is present.

In [ ]:
print(json.dumps(eda.correlation(master), indent=2))

In [ ]:
display(Image(filename=str(FIGS / 'corr_pearson.png')))

In [ ]:
display(Image(filename=str(FIGS / 'corr_spearman.png')))

In [ ]:
display(Image(filename=str(FIGS / 'corr_with_target.png')))

# Time-aware analysis — when does the signal emerge? (STT 38, RQ1)

This is the analysis specific to a time-aware problem. The left panel shows the mean trajectory of each feature by class across checkpoints; the right panel tracks **|Cohen's d| per checkpoint** — i.e. how strongly each feature separates the classes as the course progresses. `n_days_active` already exceeds the large-effect threshold (d ≥ 0.8) at **10%**; `mean_score_to_date`, `n_assessments_submitted` and `weighted_score_to_date` cross it by **20%**. The behavioural signal is therefore actionable from ~20–40% of course length, grounding the RQ1 checkpoint schedule in evidence rather than convention.

In [ ]:
print(json.dumps(eda.time_aware(checkpoints), indent=2))
display(pd.read_csv(TABLES / 'discrimination_by_checkpoint.csv', index_col=0))

In [ ]:
display(Image(filename=str(FIGS / 'time_mean_trajectory.png')))

In [ ]:
display(Image(filename=str(FIGS / 'time_discrimination_curve.png')))

# The Withdrawn early-warning signal (Step-0 Option A)

Under Option A, students who withdrew before a checkpoint are kept and labelled at-risk; their features reflect only pre-withdrawal activity. The data confirms this is a genuine signal rather than noise: median inactivity is 11 days for not-at-risk students, 116 for Fail and **233 for Withdrawn**, while median total clicks fall from 1,425 (not-at-risk) to 89 (Withdrawn). The activity collapse of withdrawing students is exactly what makes early detection feasible.

In [ ]:
print(json.dumps(eda.withdrawn_analysis(master), indent=2))
display(Image(filename=str(FIGS / 'withdrawn_activity_decay.png')))

# Findings and implications for modelling

1. **Early signal exists (RQ1).** Behavioural and performance features separate the classes from 10–20% of course length and strengthen monotonically; 40–60% is a robust, actionable window.
2. **Behaviour > demographics (RQ1/RQ2).** Engagement and assessment features dominate (d up to 2.5); demographic association is small (V ≤ 0.15). Modelling should centre on behavioural features; SHAP/LIME are expected to rank these highly.
3. **Mild imbalance (RQ3).** At 52.8% at-risk, aggressive resampling may add little; RQ3 will quantify SMOTE/ADASYN/class-weight against this baseline using PR-AUC/recall.
4. **Correlated features (RQ2).** Multicollinear engagement features may make explanation importance unstable — a factor the stability metric must account for.
5. **No leakage.** No feature is near-perfectly correlated with the label, and the time-aware cut removes future events; held-out estimates should be trustworthy.

# References

[1] M. Adnan et al., *IEEE Access*, 9:7519–7539, 2021.  
[2] N. Tomasevic, N. Gvozdenovic, S. Vranes, *Computers & Education*, 143:103676, 2020.  
[3] J. Kuzilek, M. Hlosta, Z. Zdrahal, *Scientific Data*, 4:170171, 2017.